# SABIO-RK Reaction 618 beta-glucosidase pilot

This notebook uses the local SABIO-RK Reaction 618 fixture and the registry-backed FungMod virtual-experiment API. It keeps the literature-curated scientific case underparameterized when enzyme concentration is missing, then runs a clearly marked exploratory homogeneous Michaelis-Menten virtual experiment that writes standard output tables.

This exploratory ensemble uses a user-supplied enzyme-concentration range.
The enzyme concentration is not curated from SABIO-RK EntryID 35622.
This remains an enzyme-only kinetic pilot, not a whole-fungus degradation model.

In [ ]:
from pathlib import Path
import csv
import os
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "data_registry" / "registry_index.yml").exists():
    ROOT = Path("..").resolve()

src_path = ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from fungal_model.api import VirtualExperiment
from fungal_model.data import load_kinetic_record
from fungal_model.registry import load_registry

In [ ]:
FUNGUS_ID = "sabiork_beta_glucosidase_source"
SUBSTRATE_ID = "cellobiose"
ENVIRONMENT_ID = "sabiork_reaction_618_selected_conditions"
ENZYME_CONCENTRATION_SYMBOL = "enzyme_concentration_beta_glucosidase"

REGISTRY_INDEX = ROOT / "data_registry" / "registry_index.yml"
KINETIC_RECORD_PATH = (
    ROOT
    / "data"
    / "kinetic_records"
    / "sabiork"
    / "case_001_reaction_618_beta_glucosidase"
    / "curated"
    / "kinetic_record.yml"
)

In [ ]:
registry = load_registry(REGISTRY_INDEX)
study = VirtualExperiment.from_registry(
    fungi=[FUNGUS_ID],
    substrates=[SUBSTRATE_ID],
    environments=[ENVIRONMENT_ID],
    registry=registry,
)
record = load_kinetic_record(KINETIC_RECORD_PATH)

{
    "source_database": record.source_database,
    "reaction_id": record.source_reaction_id,
    "selected_entry_id": record.source_kinetic_law_id,
    "enzyme": record.enzyme.name,
    "reaction": record.reaction.equation,
}

In [ ]:
report = study.preflight(mode="scientific")[0]

report.to_dict()

In [ ]:
exploratory_prior = next(
    parameter
    for parameter in registry.get_parameter_records(
        parameter_symbol=ENZYME_CONCENTRATION_SYMBOL,
        process_type="homogeneous_michaelis_menten",
    )
    if parameter.maturity == "exploratory_prior"
)

{
    "record_id": exploratory_prior.record_id,
    "maturity": exploratory_prior.maturity,
    "value": exploratory_prior.value.to_dict(),
    "exploratory_prior": exploratory_prior.provenance.get("exploratory_prior"),
}

In [ ]:
N_SAMPLES = 32
SEED = 1
OUTPUT_ROOT = Path(os.environ.get("FUNGMOD_NOTEBOOK_OUTPUT_ROOT", str(ROOT / "outputs")))
screen_output_dir = OUTPUT_ROOT / "sabiork_reaction_618_virtual_experiment"

virtual_result = study.simulate(
    mode="exploratory",
    n_samples=N_SAMPLES,
    seed=SEED,
    output_dir=screen_output_dir,
)
screen = virtual_result.screen_result
screen_output_dir = Path(virtual_result.output_directory)

{
    "preflight_status": virtual_result.preflight_reports[0].status,
    "simulated_status": screen.case_results[0].modelability_report.status,
    "sample_count": len(screen.case_results[0].samples),
    "table_files": sorted(virtual_result.tables.to_dict()) if virtual_result.tables is not None else [],
}

In [ ]:
def read_csv_rows(path, *, limit=None):
    with Path(path).open(newline="", encoding="utf-8") as handle:
        rows = list(csv.DictReader(handle))
    return rows if limit is None else rows[:limit]


sampled_parameter_rows = read_csv_rows(screen_output_dir / "sampled_parameters.csv")
[
    row
    for row in sampled_parameter_rows
    if row["symbol"] == ENZYME_CONCENTRATION_SYMBOL
][:5]

In [ ]:
final_metric_rows = read_csv_rows(screen_output_dir / "final_metrics.csv")
threshold_rows = read_csv_rows(screen_output_dir / "threshold_times.csv")
summary_metric_rows = read_csv_rows(screen_output_dir / "summary_metrics.csv")

final_metric_rows[:8]

In [ ]:
import matplotlib.pyplot as plt

enzyme_values = [
    float(row["sampled_value"])
    for row in sampled_parameter_rows
    if row["symbol"] == ENZYME_CONCENTRATION_SYMBOL
]

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(enzyme_values, bins=12)
ax.set_xscale("log")
ax.set_xlabel("enzyme_concentration_beta_glucosidase (mM)")
ax.set_ylabel("sample count")
ax.set_title("Sampled exploratory enzyme concentration")
fig.tight_layout()
fig

In [ ]:
final_glucose_values = [
    float(row["value"])
    for row in final_metric_rows
    if row["metric"] == "final_product_concentration"
]

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(final_glucose_values, bins=12)
ax.set_xlabel("final beta-D-glucose concentration (mM)")
ax.set_ylabel("sample count")
ax.set_title("Final beta-D-glucose concentration")
fig.tight_layout()
fig

In [ ]:
final_cellobiose_values = [
    float(row["value"])
    for row in final_metric_rows
    if row["metric"] == "final_substrate_remaining"
]

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(final_cellobiose_values, bins=12)
ax.set_xlabel("final cellobiose concentration (mM)")
ax.set_ylabel("sample count")
ax.set_title("Final cellobiose concentration")
fig.tight_layout()
fig

## Limitations

The SABIO-RK selected kinetic-law entry does not provide enzyme concentration, so scientific modelability remains underparameterized. The virtual experiment uses a user-supplied exploratory prior for enzyme concentration; this prior is not a literature-curated SABIO-RK value. This notebook does not model fungus growth, secretion, uptake, biomass, oxygen limitation, PET chemistry, cellulose surface morphology, or time-course validation data.